# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `<NN>` (replace with two-digit number from Brightspace)

**Team members:**
- Financial Services Pipeline Lead: `<name>`
- Travel and Hospitality Pipeline Lead: `<name>`
- Comparison and Visualization Lead (Integrator): `<name>`

**Submission filename:** `MP03_Notebook_team_<NN>.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [ ]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

In [ ]:
import json
import re
import time
from datetime import date, datetime, timedelta
import os
import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# -------------------------------------------------------------------------
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# -------------------------------------------------------------------------
USER_AGENT = "CIS3120 MP03 Team <NN> - aflinahossain.nometa@baruchmail.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# -------------------------------------------------------------------------
# Endpoints and constants
# -------------------------------------------------------------------------
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"



In [ ]:
# Configure the Anthropic API client.
# If your instructor expects the key in the notebook, paste it below.
ANTHROPIC_API_KEY = "API_KEY_HERE"

# Optional fallback for Colab Secrets or local environment variables.
# This runs only if the pasted value is blank or still the placeholder.
if not ANTHROPIC_API_KEY or ANTHROPIC_API_KEY == "PASTE_YOUR_ANTHROPIC_API_KEY_HERE":
    try:
        from google.colab import userdata
        ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
    except ImportError:
        ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not ANTHROPIC_API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY not found. Paste it above, set it in Colab Secrets, or set it as a local environment variable."
    )

client = Anthropic(api_key=ANTHROPIC_API_KEY)



In [ ]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

# Local refinements for the window experiment. The instructor-provided seed
# lists are intentionally small; without these additions, most current EDGAR
# hits are filtered out before Stage 3 and the pipeline can return 0 events.
FINANCIAL_SERVICES_TICKERS = sorted(set(FINANCIAL_SERVICES_TICKERS) | {
    # Regional/community banks that appear frequently in branch/opening hits.
    "RF", "FNB", "HBAN", "AUB", "FBK", "SSB", "EFSC", "HBT",
    "CNOB", "ACNB", "PFIS", "BKU", "CFG", "KEY", "FITB", "MTB",
})

FINANCIAL_SERVICES_PHRASES = [
    '"branch opening"',
    '"new branch"',
    '"opened branch"',
    '"branch closure"',
    '"branch closing"',
    '"branch consolidation"',
    '"branch network"',
    '"financial center"',
    '"banking center"',
    '"new financial center"',
    '"new banking center"',
    '"office relocation"',
    '"new office"',
    '"regional office"',
    '"regional headquarters"',
    '"operations center"',
    '"operations hub"',
    '"service center"',
    '"data center"',
    '"new location"',
]

TRAVEL_HOSPITALITY_TICKERS = sorted(set(TRAVEL_HOSPITALITY_TICKERS) | {
    # Casinos, resorts, hotels, and hospitality REITs appearing in current hits.
    "CZR", "FLL", "FUN", "PENN", "PK", "SHO", "SOHO", "AHT",
    "GLPI", "VICI", "RRR", "BALY", "PLYA", "LIND", "VENU",
})

# Put more targeted phrases first so broad phrases such as "new property" do
# not fill the max_filings cap with unrelated real-estate results.
TRAVEL_HOSPITALITY_PHRASES = [
    '"hotel opening"',
    '"resort opening"',
    '"property opening"',
    '"new hotel"',
    '"new resort"',
    '"hotel expansion"',
    '"brand conversion"',
    '"new route"',
    '"new gateway"',
    '"new terminal"',
    '"airport lounge"',
    '"new flights"',
    '"grand opening"',
    '"new property"',
] + [phrase for phrase in TRAVEL_HOSPITALITY_PHRASES if phrase not in {
    '"hotel opening"', '"resort opening"', '"property opening"',
    '"new hotel"', '"new resort"', '"hotel expansion"',
    '"brand conversion"', '"new route"', '"new gateway"',
    '"new terminal"', '"airport lounge"', '"new flights"',
    '"grand opening"', '"new property"',
}]

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")



---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [ ]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [ ]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [ ]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [ ]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [ ]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [ ]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers.
    Implementation hint: each EDGAR hit has hit["_source"]["tickers"];
    match case-insensitively and return only matching hits.
    Parameters
    ----------
    candidates : list[dict]
        EDGAR hits as returned by search_edgar_all_phrases.
    ticker_list : list[str]
        Tickers to retain (e.g., FINANCIAL_SERVICES_TICKERS).
    Returns
    -------
    list[dict]
        The subset of candidates whose tickers intersect ticker_list.
    """
    ticker_set = {ticker.upper() for ticker in ticker_list}
    filtered = []

    for hit in candidates:
        source = hit.get("_source", {})
        hit_tickers = []

        # Try structured ticker fields first when EDGAR supplies them.
        for field in ["tickers", "ticker", "symbols", "symbol"]:
            value = source.get(field, [])
            if isinstance(value, str):
                hit_tickers.append(value)
            elif isinstance(value, list):
                hit_tickers.extend(value)

        # EDGAR full-text hits often put tickers only in display_names, e.g.:
        # "Company Name  (ABC, ABC-PD)  (CIK 000...)".
        display_names = source.get("display_names", [])
        if isinstance(display_names, str):
            display_names = [display_names]

        for name in display_names:
            matches = re.findall(r"\(([^()]+)\)", str(name))
            for match in matches:
                if match.upper().startswith("CIK"):
                    continue
                for possible_ticker in match.split(","):
                    possible_ticker = possible_ticker.strip().upper()
                    # Accept common tickers and share-class tickers while
                    # still rejecting CIK numbers and company-name fragments.
                    if re.match(r"^[A-Z][A-Z0-9.-]{0,7}$", possible_ticker):
                        hit_tickers.append(possible_ticker)

        hit_ticker_set = {str(ticker).upper() for ticker in hit_tickers}
        if hit_ticker_set & ticker_set:
            filtered.append(hit)

    return filtered



In [36]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice.

    Returns geocoded events with an "industry" field added to each record.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    ticker_list : list[str]
        Industry-specific ticker list.
    phrase_list : list[str]
        Industry-specific search-phrase list.
    window_days : int
        Length of the EDGAR date window (e.g., 30, 60, 90, 180, 360).

    Returns
    -------
    list[dict]
        One dict per geocoded location event, with the "industry" field set
        to industry_label. Records that fail classification or geocoding are
        excluded from the return value.
    """

    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    candidates = search_edgar_all_phrases(
        phrases=phrase_list,
        start_date=start_date,
        end_date=end_date,
        max_filings=500,
    )

    filtered_candidates = filter_candidates_by_tickers(
        candidates=candidates,
        ticker_list=ticker_list,
    )

    print(
        f"{industry_label} ({window_days} days): "
        f"{len(candidates)} raw candidates, "
        f"{len(filtered_candidates)} after ticker filter"
    )

    events = []

    for hit in filtered_candidates:
        exhibit_url = build_exhibit_url(hit)
        exhibit_text, fetched_url = fetch_exhibit_text(hit)

        filing = {
            "hit": hit,
            "url": exhibit_url,
            "text": exhibit_text,
            "fetch_status": "ok",
        }

        extracted = extract_with_claude(filing)

        if not extracted.get("is_location_event"):
            continue

        city = extracted.get("city")
        state = extracted.get("state")

        if not city:
            continue

        coordinates = geocode_location(city, state)

        if not coordinates:
            continue

        latitude, longitude = coordinates

        record = {
            **extracted,
            "industry": industry_label,
            "latitude": latitude,
            "longitude": longitude,
            "filing_url": fetched_url,
            "fetch_status": "ok",
        }

        events.append(record)

    print(f"{industry_label} ({window_days} days): {len(events)} geocoded events")
    return events



In [37]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    window_days : int
        One of 30, 60, 90, 180, 360.
    candidate_count : int
        Length of filtered candidate list before Stage 3.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Approximate API spend for this trial; sum of input + output token
        cost at Haiku 4.5 pricing ($1/M input, $5/M output).

    Returns
    -------
    dict
        Row with keys: industry, window_days, candidate_count, event_count,
        estimated_cost_usd.
    """
    # TODO: implement this function.
    return {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": candidate_count,
        "event_count": event_count,
        "estimated_cost_usd": round(float(estimated_cost_usd), 6),
}

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [38]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


In [39]:
financial_events = run_industry_pipeline(
    industry_label="Financial Services",
    ticker_list=FINANCIAL_SERVICES_TICKERS,
    phrase_list=FINANCIAL_SERVICES_PHRASES,
    window_days=360,
)

print(len(financial_events))

Financial Services (360 days): 500 raw candidates, 84 after ticker filter


AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CbAoh8XRsPb7qRTS6t99m'}

In [ ]:
financial_df = pd.DataFrame(financial_events)
financial_df

In [ ]:
financial_cost = (
    financial_df["input_tokens"].sum() * 1 / 1_000_000
    + financial_df["output_tokens"].sum() * 5 / 1_000_000
)

fs_row_360 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=360,
    candidate_count=84,  
    event_count=len(financial_events),
    estimated_cost_usd=financial_cost,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([fs_row_360])],
    ignore_index=True,
)

window_results

In [ ]:
financial_df[
    [
        "industry",
        "event_type",
        "city",
        "state",
        "summary",
        "latitude",
        "longitude",
        "filing_url",
        "input_tokens",
        "output_tokens",
    ]
]

In [ ]:
financial_df = pd.DataFrame(financial_events)

financial_map_df = financial_df[
    [
        "industry",
        "event_type",
        "city",
        "state",
        "summary",
        "latitude",
        "longitude",
        "filing_url",
        "input_tokens",
        "output_tokens",
    ]
]

financial_map_df.to_csv("financial_services_events_team_09.csv", index=False)

financial_map_df

In [ ]:
travel_events = run_industry_pipeline(
    industry_label="Travel and Hospitality",
    ticker_list=TRAVEL_HOSPITALITY_TICKERS,
    phrase_list=TRAVEL_HOSPITALITY_PHRASES,
    window_days=360,
)

In [ ]:
print(len(travel_events))

In [ ]:
pd.DataFrame(travel_events)

In [ ]:
###Lines below are experimental.

In [ ]:
travel_events_360 = run_industry_pipeline(
    industry_label="Travel and Hospitality",
    ticker_list=TRAVEL_HOSPITALITY_TICKERS,
    phrase_list=TRAVEL_HOSPITALITY_PHRASES,
    window_days=360,
)

print(len(travel_events_360))
pd.DataFrame(travel_events_360)

In [ ]:
pd.DataFrame(travel_events_360)[[
    "event_type",
    "city",
    "state",
    "summary",
    "industry"
]]

In [ ]:
travel_df = pd.DataFrame(travel_events_360)

drop_indexes = [2, 5, 6, 8, 9, 10, 11, 15]

travel_df_clean = travel_df.drop(index=drop_indexes).reset_index(drop=True)

travel_df_clean[[
    "event_type",
    "city",
    "state",
    "summary",
    "industry"
]]

In [ ]:
travel_df_clean.to_csv("travel_hospitality_events_clean.csv", index=False)

In [ ]:
len(travel_df_clean)

In [ ]:
row_360 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=360,
    candidate_count=59,
    event_count=len(travel_df_clean),
    estimated_cost_usd=0.31,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([row_360])],
    ignore_index=True
)

window_results

### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [ ]:
fs_events_by_window = {}

for window_days in [30, 60, 90, 180, 360]:
    print(f"Running Financial Services trial for {window_days} days...")

    fs_events = run_industry_pipeline(
        industry_label="Financial Services",
        ticker_list=FINANCIAL_SERVICES_TICKERS,
        phrase_list=FINANCIAL_SERVICES_PHRASES,
        window_days=window_days,
    )

    fs_events_by_window[window_days] = fs_events

    fs_df = pd.DataFrame(fs_events)

    if len(fs_df) > 0:
        fs_cost = (
            fs_df["input_tokens"].sum() * 1 / 1_000_000
            + fs_df["output_tokens"].sum() * 5 / 1_000_000
        )
    else:
        fs_cost = 0.00

    fs_row = summarize_window_trial(
        industry_label="Financial Services",
        window_days=window_days,
        candidate_count=0,  
        event_count=len(fs_events),
        estimated_cost_usd=fs_cost,
    )

    window_results = pd.concat(
        [window_results, pd.DataFrame([fs_row])],
        ignore_index=True,
    )

    display(window_results)

    if len(fs_events) >= 8:
        print(f"Financial Services reached 8 events at {window_days} days.")
        break


### 4.2 Window trials — Travel and Hospitality

In [ ]:
# 4.2 Window trials — Travel and Hospitality

travel_df = pd.DataFrame(travel_events)

travel_cost = (
    travel_df["input_tokens"].sum() * 1 / 1_000_000
    + travel_df["output_tokens"].sum() * 5 / 1_000_000
)

th_row_360 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=360,
    candidate_count=60,  
    event_count=len(travel_events),
    estimated_cost_usd=travel_cost,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([th_row_360])],
    ignore_index=True,
)

window_results

### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [ ]:
import pandas as pd

CHOSEN_WINDOW_DAYS = 360

# CSVs are in the same folder as this notebook
financial_df = pd.read_csv("financial_services_events_team_09.csv")
travel_df = pd.read_csv("travel_hospitality_events_clean.csv")

fs_events = financial_df.to_dict("records")
th_events = travel_df.to_dict("records")

all_events = fs_events + th_events

print(f"Chosen window:            {CHOSEN_WINDOW_DAYS} days")
print(f"Financial Services:       {len(fs_events)} events")
print(f"Travel and Hospitality:   {len(th_events)} events")
print(f"Total:                    {len(all_events)} events")

---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [43]:
import html
import folium


INDUSTRY_COLORS = {
    "Financial Services": "blue",
    "Travel and Hospitality": "green",
}


EVENT_ICONS = {
    "opening": "plus",
    "closing": "minus",
    "relocation": "exchange",
    "expansion": "arrow-up",
    "other": "info-circle",
}

m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

for event in all_events:
    
    lat = event.get("latitude")
    lon = event.get("longitude")

    if pd.isna(lat) or pd.isna(lon):
        continue

    industry = event.get("industry", "Unknown")
    event_type = event.get("event_type", "other")
    city = event.get("city", "")
    state = event.get("state", "")
    summary = event.get("summary", "")
    filing_url = event.get("filing_url", event.get("url", ""))

    popup_html = f"""
    <b>Industry:</b> {html.escape(str(industry))}<br>
    <b>Event type:</b> {html.escape(str(event_type))}<br>
    <b>Location:</b> {html.escape(str(city))}, {html.escape(str(state))}<br>
    <b>Summary:</b> {html.escape(str(summary))}<br>
    <a href="{html.escape(str(filing_url))}" target="_blank">SEC filing</a>
    """

    marker = folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=350),
        icon=folium.Icon(
            color=INDUSTRY_COLORS.get(industry, "gray"),
            icon=EVENT_ICONS.get(event_type, "info-circle"),
            prefix="fa",
        ),
    )

    marker.add_to(m)

m

### Export the map to `maps/mp03_map_team_<NN>.html`

In [46]:
import os

os.makedirs("maps", exist_ok=True)

m.save("maps/mp03_map_team_09.html")

print(os.listdir("maps"))

['mp03_map_team_09.html']


In [47]:
from pathlib import Path

OUTPUT_PATH = "../maps/mp03_map_team_09.html"

Path("../maps").mkdir(exist_ok=True)

m.save(OUTPUT_PATH)

print(f"Map saved to {OUTPUT_PATH}")

Map saved to ../maps/mp03_map_team_09.html


---

## 6. Methodology

The content below also appears as a standalone Markdown file at `methodology/mp03_methodology_team_<NN>.md`. Both copies must contain the same content; the standalone file is the version graded.

### 6.1 Ticker-list rationale

*TODO: For each industry, justify any modifications to the seeded ticker list. Identify what the seeded list undercounts or overcounts and explain how your changes address those limitations.*

### 6.2 Search-phrase rationale

*TODO: For each industry, justify any modifications to the seeded phrase list. Note any phrases that returned high-volume false positives or missed event categories the team considered important.*

### 6.3 Window-experiment results

*TODO: Insert the populated `window_results` table here (as Markdown) and explain why the chosen window is appropriate. Address the cost ceiling explicitly.*

### 6.4 Stage 3 classification quality per industry

*TODO: For each industry, document observed precision and any patterns in the Stage 3 classifications (false positives, false negatives, ambiguous cases). Use small numerical examples where possible.*

### 6.5 Limitations

*TODO: Identify limitations the team encountered and discuss how each affects the comparative reflection.*

---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_<NN>.md`.

*TODO: Write the comparative reflection here. Mere description of the maps does not earn full credit; the reflection must offer substantive interpretation grounded in the underlying business economics and address limitations honestly.*

---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.

In [54]:
import os

os.makedirs("maps", exist_ok=True)

m.save("maps/mp03_map_team_09.html")

print("saved map")
print(os.listdir("maps"))

saved map
['mp03_map_team_09.html']


In [55]:
import os
os.getcwd()

'/Users/mariam/notebooks'

In [56]:
os.listdir("maps")

['mp03_map_team_09.html']

In [57]:
import os

os.makedirs("maps", exist_ok=True)

m.save("/Users/mariam/cis3120-spring2026/maps/mp03_map_team_09.html")